**On va tenter de prédire les paramètres à passer à Prophet pour chacune des séries temporelles en fonction de leurs caractéristiques générales : moyenne, tendance, assymétrie, aplatissement, variance, entropie, etc.**

Le prnicipe : On connaît déjà grâce au notebook autoforecast_time_series les paramètres optimaux à passer à Prophet, et maintenant on voudrait que le programme s'exécute plus vite, qu'on n'ait pas à attendre deux heures et demie à chaque fois, donc on va entraîner un modèle qui connaît déjà les résultats pour qu'il ajuste une fonction de décision qui permettra de choisir les hyperparamètres de Prophet sans avoir à boucler dessus, simplement en regardant l'allure de la série. Une fois que ce sera fait, on pourra ajouter plein de nouvelles séries temporelles, qui pourront être prédites beaucoup plus rapidement, sans avoir à passer par un balayage complet de tous les hyperparamètres possibles. 

In [9]:
# On importe les bibliothèques qui vont bien

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

In [10]:
# Import et nettoyage du tableau de données qu'on va utiliser (on fait exactement la même chose que pour le clustering)

series_temp_features = pd.read_excel("series_temp_features.xlsx")

print(series_temp_features.columns)

# Correction des NaN sinon le clustering ne peut pas fonctionner
i = 0
for colonne in series_temp_features.columns :
    for index, valeur in series_temp_features[colonne].items() :
        if pd.isna(valeur) :
            i += 1
            print(f"NaN trouvé dans la colonne '{colonne}', à l'index {index}")
print(f'Il y a {i} erreurs à corriger')

series_temp_features['std_normalise_first_half'] = series_temp_features['std_normalise_first_half'].fillna(0)
series_temp_features['std_normalise_second_half'] = series_temp_features['std_normalise_second_half'].fillna(0)
series_temp_features['std_normalise_ecart'] = series_temp_features['std_normalise_ecart'].fillna(abs(series_temp_features['std_normalise_first_half']-series_temp_features['std_normalise_second_half']))
series_temp_features = series_temp_features[series_temp_features.isnull().sum(axis=1) < 7]
series_temp_features['mode_saisonnalite'] = series_temp_features['mode_saisonnalite'].fillna('aucune')
series_temp_features['id_methode'] = series_temp_features['id_methode'].fillna('echec_total')
valeur_max = series_temp_features['ratio_debut_fin'].max(skipna=True)
series_temp_features['ratio_debut_fin'] = series_temp_features['ratio_debut_fin'].fillna(valeur_max + 1)

j = 0
for colonne in series_temp_features.columns :
    for index, valeur in series_temp_features[colonne].items() :
        if pd.isna(valeur) :
            j += 1
            print(f"NaN trouvé dans la colonne '{colonne}', à l'index {index}")
print(f'Il n\'y a plus que {j} erreurs à corriger (normalement c\'est 0)')

# On extrait un tableau qui ne contient que les variables qui vont nous servir à prédire
features_input = series_temp_features.drop(columns=[
    'saisonnalite', 
    'mode_saisonnalite', 
    'mcmc_utilise', 
    'changepoint_prior_scale', 
    'id_methode', 
    'rmsse',
    'nom_serie'  # identifiant à garder à part si besoin
])

# Standardisation des données
features_standardisees = StandardScaler().fit_transform(features_input)

Index(['nom_serie', 'groupe_id', 'longueur_serie', 'mean_abs_diff',
       'std_normalise', 'std_normalise_first_half',
       'std_normalise_second_half', 'std_normalise_ecart', 'moyenne1',
       'moyenne2', 'ecart_moyennes_normalise', 'skewness', 'kurtosis',
       'moyenne_debut', 'moyenne_fin', 'ratio_debut_fin',
       'ecart_interquantile', 'autocorr', 'autocorrelation_saisonniere',
       'turning_points', 'mediane', 'tendance', 'adf_pvalue',
       'fft_dominant_freq', 'fft_dominant_amp', 'fft_dominant_phase',
       'shannon_entropy', 'mcmc_utilise', 'saisonnalite', 'mode_saisonnalite',
       'changepoint_prior_scale', 'rmsse', 'id_methode', 'surajustement'],
      dtype='object')
NaN trouvé dans la colonne 'std_normalise_first_half', à l'index 406
NaN trouvé dans la colonne 'std_normalise_first_half', à l'index 479
NaN trouvé dans la colonne 'std_normalise_first_half', à l'index 484
NaN trouvé dans la colonne 'std_normalise_second_half', à l'index 452
NaN trouvé dans la col

In [11]:
# Random Forest pour prédire changepoint_prior_scale

X = features_standardisees
y = series_temp_features['changepoint_prior_scale']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Performance du modèle :")
print(f" - RMSE : {mean_squared_error(y_test, y_pred) ** 0.5:.4f}")
print(f" - R² : {r2_score(y_test, y_pred):.4f}")

Performance du modèle :
 - RMSE : 0.0295
 - R² : 0.5675


RMSE de 0.0295 : C'est beaucoup vu que les valeurs sont petites...

R2 de 0.5675 : le modèle explique 56,75 % de la variance c'est pas mal !

In [12]:
# Prédiction de MCMC avec Random Forest
y_class = series_temp_features['mcmc_utilise'].astype(int)  # assure-toi que c’est bien 0/1

X_train, X_test, y_train, y_test = train_test_split(X, y_class, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Classification : MCMC utilisé")
print(f" - Précision : {accuracy_score(y_test, y_pred):.3f}")
print(" - Matrice de confusion :\n", confusion_matrix(y_test, y_pred))
print(" - Rapport complet :\n", classification_report(y_test, y_pred))

Classification : MCMC utilisé
 - Précision : 0.648
 - Matrice de confusion :
 [[38 20]
 [23 41]]
 - Rapport complet :
               precision    recall  f1-score   support

           0       0.62      0.66      0.64        58
           1       0.67      0.64      0.66        64

    accuracy                           0.65       122
   macro avg       0.65      0.65      0.65       122
weighted avg       0.65      0.65      0.65       122



Précision 0.659 : 65,9% du temps on a prédit la bonne chose

Matrice de confusion : en 0,0 on a que 48 séries sans MCMC on été prédites comme sans MCMC, en 1,1 on a que 35 séries avec MCMC ont été prédites avec MCMC, en 0,1 on a que 15 séries ont été prédites avec MCMC alors qu'elle n'en ont pas besoin en réalité, et vice versa en 1,0. 

Classe 0 : MCMC pas utilisé

Précision : 0.63 : sur tous les cas où le modèle a prédit "pas de MCMC", 63 % étaient corrects.

Recall (rappel) : 0.76 : le modèle a correctement trouvé 76 % des cas où il n’y avait pas de MCMC.

F1-score : 0.69 : équilibre entre précision et rappel.

Classe 1 : MCMC utilisé

Précision : 0.70 : 70 % des prédictions "MCMC utilisé" étaient bonnes.

Recall : 0.56 : le modèle n’a détecté que 56 % des vrais cas de MCMC.

F1-score : 0.62

In [ ]:
# Encodage des labels
le = LabelEncoder()
y_multi = le.fit_transform(series_temp_features['mode_saisonnalite'])

# On sépare avec une partie des données pour l'entraînement et une partie des données pour le test
X_train, X_test, y_train, y_test = train_test_split(X, y_multi, test_size=0.2, random_state=42)

clf_multi = RandomForestClassifier(n_estimators=100, random_state=42)
clf_multi.fit(X_train, y_train)

# Évaluation
y_pred = clf_multi.predict(X_test)

print("Classification : mode de saisonnalité")
print(f" - Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(" - Matrice de confusion :\n", confusion_matrix(y_test, y_pred))
print(" - Rapport :\n", classification_report(y_test, y_pred, target_names=le.classes_))

Classification : mode de saisonnalité
 - Accuracy : 0.557
 - Matrice de confusion :
 [[10 10 10]
 [ 2 33 13]
 [ 3 16 25]]
 - Rapport :
                 precision    recall  f1-score   support

      additive       0.67      0.33      0.44        30
        aucune       0.56      0.69      0.62        48
multiplicative       0.52      0.57      0.54        44

      accuracy                           0.56       122
     macro avg       0.58      0.53      0.53       122
  weighted avg       0.57      0.56      0.55       122



Même chose ici. 

**Conclusion : la régression supervisée ne fonctionne pas ! Les taux d'erreurs sont énormes dans quasiment tous les cas, il semblerait que le seul moyen de vraiment savoir avec un bon niveau de certitude quels sont les meilleurs paramètres à donner à Prophet soit de tous les balayer et de comparer les RMSE... Ca prend du temps à l'exécution, mais au moins c'est fiable. Rappelons qu'en moyenne, les modèles obtenus par balayage systématique des hyperparamètres prédisent en moyenne 41% mieux que les modèles naïfs (c'est à dire prévsion pour la date suivante = valeur actuelle) !**